# Data Transformation

In [1]:
import os

In [2]:
%pwd

'd:\\AI - Courses\\MLOPs\\wine_quality_project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\AI - Courses\\MLOPs\\wine_quality_project'

In [5]:
import pandas as pd

In [14]:
df = pd.read_csv('artifacts/data_ingestion/winequality-red.csv')
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [16]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [17]:
from src.wine_quality_project.constants import *
from src.wine_quality_project.utils.common import read_yaml, create_directories

In [18]:
class ConfigurationManager:
    def __init__(self,
                 conifg_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_fielpath=SCHEMA_FILE_PATH):
        self.config=read_yaml(conifg_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_fielpath)
        
        create_directories([self.config.artifacts_root])


    def get_data_transformation_config(self)-> DataTransformationConfig:
        config=self.config.data_transformation

        create_directories([config.root_dir])
        
        data_transformation_config=DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path
        )
        
        return data_transformation_config


In [19]:
import os
from src.wine_quality_project import logger
from sklearn.model_selection import train_test_split

In [20]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config=config

    def train_test_split(self):
        data = pd.read_csv(self.config.data_path)
        train, test = train_test_split(data, test_size=0.25)
        train.to_csv(os.path.join(self.config.root_dir, 'train.csv'), index=False)
        test.to_csv(os.path.join(self.config.root_dir, 'test.csv'), index=False)
        logger.info('splitting data into train and test sets')
        logger.info(train.shape)
        logger.info(test.shape)
        print(train.shape)
        print(test.shape)

In [21]:
try:
    config = ConfigurationManager()
    data_transformation_config=config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_split()
except Exception as e:
    raise e


[2025-06-19 14:03:09,483: INFO: common: yaml file: config\config.yaml loaded successfuly]
[2025-06-19 14:03:09,489: INFO: common: yaml file: params.yaml loaded successfuly]
[2025-06-19 14:03:09,495: INFO: common: yaml file: schema.yaml loaded successfuly]
[2025-06-19 14:03:09,497: INFO: common: created directories at: artifacts]
[2025-06-19 14:03:09,498: INFO: common: created directories at: artifacts/data_transformation]
[2025-06-19 14:03:09,540: INFO: 307722839: splitting data into train and test sets]
[2025-06-19 14:03:09,544: INFO: 307722839: (1199, 12)]
[2025-06-19 14:03:09,546: INFO: 307722839: (400, 12)]
(1199, 12)
(400, 12)


In [23]:
with open(Path('artifacts/data_validation/status.txt'), 'r') as file:
    status = file.read().split(' ')[-1]
    print(status)

True
